# HW4: Neural Retrieval and Re-ranking

Используем два датасета: **WikIR1k** и **MIRAGE**.

Что делаем:
1. Считаем **dense retrieval** по всему корпусу через bi-encoder.
2. Делаем **re-ranking**: сначала BM25 отбирает кандидатов, потом cross-encoder их пересортировывает.
3. Строим **mixture**: смешиваем BM25 и dense score.

Для сравнения также берем baseline из HW2 (чистый BM25) и модель LTR из HW3 (CatBoost на лексических и dense-признаках).

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

HW4_ROOT = Path.cwd()
if HW4_ROOT.name != "hw4":
    HW4_ROOT = HW4_ROOT / "hw4"

REPO_ROOT = HW4_ROOT.parent
sys.path.insert(0, str(REPO_ROOT.resolve()))

from src import (
    apply_catboost_ltr,
    apply_mixture,
    build_mirage_bm25_run,
    build_mirage_collection,
    evaluate_run,
    extract_lexical_features,
    get_best_device,
    get_local_mirage_paths,
    load_local_mirage_split,
    load_wikir_bm25_run,
    load_wikir_split,
    make_ranked_run,
    run_dense_retrieval,
    score_candidate_pairs_biencoder,
    score_candidate_pairs_cross_encoder,
    top_k_per_query,
    train_catboost_ltr,
    tune_alpha,
)

/opt/homebrew/Caskroom/miniforge/base/envs/ir_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_ROOT = REPO_ROOT / "data"
RUNS_DIR = REPO_ROOT / "runs" / "hw4"
EMBED_DIR = DATA_ROOT / "embeddings"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

RERANK_K_VALUES = [10, 20, 50, 100]
CANDIDATE_K = 1000
ALPHA_GRID = np.linspace(0.0, 1.0, 21)

BATCH_SIZE = 32
CROSS_BATCH_SIZE = 32
DEVICE = get_best_device()

DENSE_MODELS = {
    "all-MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "multi-qa-MiniLM-L6-cos-v1": "sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
}
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"
BIENC_MODEL = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"

print(f"Device: {DEVICE}")

Device: mps


## Данные

Используем два датасета:

- **WikIR** — коллекция WikIR (`en1k` subset).
  Сплит `test` используем для итоговой оценки, а `training` — для подбора `alpha` в mixture и для обучения CatBoost LTR.

- **MIRAGE** — QA-датасет (`nlpai-lab/mirage`).
  В нем нет общего корпуса документов: у каждого запроса есть свой `doc_pool` с кандидатами (`doc_name`, `doc_chunk`, `support`).

Для `MIRAGE` готового train/test split нет, поэтому мы делаем его сами в `scripts/prepare_mirage_split.py`.
Делим данные так, чтобы похожие источники не попадали одновременно в train и test.

Функция `build_mirage_collection` переводит MIRAGE в обычный IR-формат: `docs`, `queries`, `qrels`.

- документ задаем как строку `"title. chunk"`
- `doc_id` строим как md5 от текста
- `relevance` берем из `support`

BM25 используем по-разному:

- для **WikIR** берем готовые файлы `BM25.res`
- для **MIRAGE** считаем BM25 отдельно внутри `doc_pool` каждого запроса


In [3]:
wikir_docs, wikir_test_queries, wikir_test_qrels = load_wikir_split(DATA_ROOT, split="test")
_, wikir_train_queries, wikir_train_qrels = load_wikir_split(DATA_ROOT, split="training")
wikir_bm25_test = load_wikir_bm25_run(DATA_ROOT, split="test")
wikir_bm25_train = load_wikir_bm25_run(DATA_ROOT, split="training")

assert (get_local_mirage_paths(DATA_ROOT)["split_dir"] / "hf").exists(), (
    "MIRAGE split not found. Run from repo root: python scripts/prepare_mirage_split.py"
)

mirage_test_dataset = load_local_mirage_split(DATA_ROOT, split="test")
mirage_test_docs, mirage_test_queries, mirage_test_qrels = build_mirage_collection(mirage_test_dataset)
mirage_bm25_test = build_mirage_bm25_run(mirage_test_dataset, top_k=100)

## 1. Dense Retrieval

Здесь используем **bi-encoder**. Он кодирует запрос и документ отдельно, а потом сравнивает их векторы по dot product / cosine.

Плюс такого подхода в том, что он быстрый: эмбеддинги документов можно посчитать один раз и сохранить, а для нового запроса остается только получить его вектор и найти ближайшие документы.

Минус в том, что запрос и документ обрабатываются по отдельности, поэтому модель обычно слабее, чем cross-encoder.

Две модели:
- `all-MiniLM-L6-v2` — быстрый общий baseline.
- `multi-qa-MiniLM-L6-cos-v1` — специализирована под question → passage retrieval, ожидаемо лучше на задаче.

In [4]:
DENSE_COLLECTIONS = {
    "WikIR test": (wikir_docs, wikir_test_queries, wikir_test_qrels),
    "MIRAGE test": (mirage_test_docs, mirage_test_queries, mirage_test_qrels),
}


def run_dense():
    rows = []
    for dataset_name, (docs_df, queries_df, qrels_df) in DENSE_COLLECTIONS.items():
        for short_name, model_name in DENSE_MODELS.items():
            start = time.perf_counter()
            run_df = run_dense_retrieval(
                model_name=model_name,
                docs_df=docs_df,
                queries_df=queries_df,
                top_k=20,
                batch_size=BATCH_SIZE,
                device=DEVICE,
                run_name=f"dense_{short_name}",
                cache_dir=EMBED_DIR,
            )
            elapsed = time.perf_counter() - start
            run_df.to_csv(
                RUNS_DIR / f"dense__{dataset_name.lower().replace(' ', '_')}__{short_name}.csv",
                index=False,
            )
            rows.append({
                "dataset": dataset_name,
                "model": short_name,
                "seconds": round(elapsed, 1),
                **evaluate_run(run_df, qrels_df),
            })
    return pd.DataFrame(rows).sort_values(["dataset", "nDCG@20"], ascending=[True, False]).reset_index(drop=True)

In [5]:
dense_df = run_dense()
display(dense_df)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9397.72it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10244.81it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6543.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+-

,dataset,model,seconds,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,multi-qa-MiniLM-L6-cos-v1,25.1,0.638045,0.117569,0.059379,0.769473,0.827687
1,MIRAGE test,all-MiniLM-L6-v2,20.1,0.622193,0.118098,0.059313,0.757662,0.819441
2,WikIR test,all-MiniLM-L6-v2,1470.6,0.640000,0.185000,0.126500,0.121602,0.357409
3,WikIR test,multi-qa-MiniLM-L6-cos-v1,1625.1,0.630000,0.184000,0.121500,0.122343,0.354670


## 2. Re-ranking

Здесь используем **cross-encoder**. Он получает пару `[query, doc]` целиком и сразу выдает один score релевантности.

Обычно он работает точнее, чем bi-encoder, но намного медленнее. Поэтому считать его по всему корпусу неудобно.

Стандартная схема такая:
BM25 сначала берет top-k кандидатов, а потом cross-encoder их пересортировывает.

В этом пункте используем `cross-encoder/ms-marco-MiniLM-L6-v2` и смотрим, как меняется качество при разных `k`.

In [6]:
RERANK_COLLECTIONS = {
    "WikIR test": (wikir_docs, wikir_test_queries, wikir_test_qrels, wikir_bm25_test),
    "MIRAGE test": (mirage_test_docs, mirage_test_queries, mirage_test_qrels, mirage_bm25_test),
}


def run_reranking(k_values=None):
    if k_values is None:
        k_values = RERANK_K_VALUES

    rows = []
    for dataset_name, (docs_df, queries_df, qrels_df, bm25_run) in RERANK_COLLECTIONS.items():
        max_k = int(bm25_run.groupby("query_id").size().min())
        effective_ks = [k for k in sorted(set(k_values)) if k <= max_k] or [max_k]
        candidates = top_k_per_query(bm25_run, max(effective_ks))

        start = time.perf_counter()
        scored = score_candidate_pairs_cross_encoder(
            model_name=CE_MODEL,
            queries_df=queries_df,
            docs_df=docs_df,
            candidates_df=candidates,
            batch_size=CROSS_BATCH_SIZE,
            device=DEVICE,
        )
        elapsed = time.perf_counter() - start

        for k in effective_ks:
            reranked = make_ranked_run(
                top_k_per_query(scored, k),
                score_col="rerank_score",
                run_name=f"rerank_{dataset_name.lower().replace(' ', '_')}_k{k}",
                top_k=k,
            )
            reranked.to_csv(RUNS_DIR / f"rerank__{dataset_name.lower().replace(' ', '_')}__k{k}.csv", index=False)
            rows.append({
                "dataset": dataset_name,
                "k": k,
                "pairs_per_sec": round(len(scored) / elapsed, 1) if elapsed > 0 else float("nan"),
                **evaluate_run(reranked, qrels_df),
            })
    return pd.DataFrame(rows).sort_values(["dataset", "k"]).reset_index(drop=True)

In [7]:
rerank_df = run_reranking()
display(rerank_df)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13447.80it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7134.67it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 119/119 [00:21<00:00,  5.47it/s]


,dataset,k,pairs_per_sec,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,5,146.6,0.842801,0.120079,0.06004,0.902459,0.929068
1,WikIR test,10,185.1,0.490000,0.127000,0.06350,0.106218,0.236378
2,WikIR test,20,185.1,0.530000,0.146000,0.09450,0.129518,0.281823
3,WikIR test,50,185.1,0.570000,0.155000,0.10950,0.140766,0.315642
4,WikIR test,100,185.1,0.590000,0.168000,0.11650,0.147547,0.330881


## 3. Mixture Model

Смешиваем два score:

`alpha * BM25_norm + (1 - alpha) * cosine_similarity`

Здесь `BM25_norm` — это BM25 после min-max нормализации по запросу, а `cosine_similarity` — dense score от bi-encoder.

Параметр `alpha` подбираем на `WikIR train` по метрике `nDCG@20`.

In [13]:
MIXTURE_COLLECTIONS = {
    "WikIR test": (wikir_docs, wikir_test_queries, wikir_test_qrels, wikir_bm25_test),
    "MIRAGE test": (mirage_test_docs, mirage_test_queries, mirage_test_qrels, mirage_bm25_test),
}

def run_mixture(candidate_k=CANDIDATE_K):
    train_scored = score_candidate_pairs_biencoder(
        model_name=BIENC_MODEL,
        queries_df=wikir_train_queries,
        docs_df=wikir_docs,
        candidates_df=top_k_per_query(wikir_bm25_train, candidate_k),
        batch_size=BATCH_SIZE,
        device=DEVICE,
        output_col="dense_score",
        cache_dir=EMBED_DIR,
    )
    best_alpha, tuning_df = tune_alpha(
        candidates_df=train_scored,
        qrels_df=wikir_train_qrels,
        alphas=ALPHA_GRID,
        top_k=20,
    )

    rows = []
    for dataset_name, (docs_df, queries_df, qrels_df, bm25_run) in MIXTURE_COLLECTIONS.items():
        scored = score_candidate_pairs_biencoder(
            model_name=BIENC_MODEL,
            queries_df=queries_df,
            docs_df=docs_df,
            candidates_df=top_k_per_query(bm25_run, candidate_k),
            batch_size=BATCH_SIZE,
            device=DEVICE,
            output_col="dense_score",
            cache_dir=EMBED_DIR,
        )
        tag = dataset_name.split()[0].lower()
        _, mixture_run = apply_mixture(
            scored, alpha=best_alpha, top_k=20,
            run_name=f"mixture_{tag}_alpha{best_alpha:.2f}",
        )
        dense_run = make_ranked_run(scored, score_col="dense_score", run_name="dense_only", top_k=20)
        bm25_run_20 = top_k_per_query(bm25_run, 20)
        mixture_run.to_csv(RUNS_DIR / f"mixture__{tag}.csv", index=False)

        for method, run in [
            ("BM25", bm25_run_20),
            ("Dense", dense_run),
            (f"Mixture \u03b1={best_alpha:.2f}", mixture_run),
        ]:
            rows.append({"dataset": dataset_name, "method": method, **evaluate_run(run, qrels_df)})

    return best_alpha, tuning_df, pd.DataFrame(rows)

In [9]:
best_alpha, alpha_tuning_df, mixture_df = run_mixture()
print(f"Best alpha = {best_alpha:.2f}")
display(alpha_tuning_df.head(5))
display(mixture_df)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 49742.47it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8001.28it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6214.23it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
---------------------

Best alpha = 0.10


,alpha,nDCG@20
0,0.10,0.306669
1,0.05,0.305688
2,0.15,0.305554
3,0.20,0.303385
4,0.25,0.301162


,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,WikIR test,BM25,0.230000,0.133000,0.09450,0.097617,0.215954
1,WikIR test,Dense,0.470000,0.148000,0.10450,0.128607,0.298972
2,WikIR test,Mixture α=0.10,0.510000,0.158000,0.10650,0.137244,0.308212
3,MIRAGE test,BM25,0.496697,0.120079,0.06004,0.675919,0.759824
4,MIRAGE test,Dense,0.663144,0.120079,0.06004,0.795332,0.849199
5,MIRAGE test,Mixture α=0.10,0.690885,0.120079,0.06004,0.810315,0.860548


## HW2 & HW3 Baseline

- **HW2**: обычный BM25 `original` используем как базовый метод для сравнения. Для `WikIR` берем готовый run, а для `MIRAGE` считаем BM25 по кандидатам из `doc_pool`.
- **HW3**: CatBoost LTR на тех же данных. В качестве признаков берем `bm25_norm`, `dense_score` и лексические фичи: `tf_sum`, `tf_max`, `idf_sum`, `idf_max`, `doc_len`, `query_len`, `matched_terms_ratio`, `min_span`.
- CatBoost обучаем на `WikIR train`, а качество смотрим на `WikIR test` и `MIRAGE test`.

In [16]:
BM25_COLLECTIONS = {
    "WikIR test": (wikir_test_qrels, wikir_bm25_test),
    "MIRAGE test": (mirage_test_qrels, mirage_bm25_test),
}

hw2_bm25_original_df = pd.DataFrame([
    {
        "dataset": dataset_name,
        "method": "HW2 BM25 original",
        **evaluate_run(bm25_run, qrels_df),
    }
    for dataset_name, (qrels_df, bm25_run) in BM25_COLLECTIONS.items()
])

display(hw2_bm25_original_df)


,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,WikIR test,HW2 BM25 original,0.230000,0.132000,0.09500,0.097476,0.216089
1,MIRAGE test,HW2 BM25 original,0.496697,0.120079,0.06004,0.675919,0.759824


In [ ]:
def run_catboost_ltr(candidate_k=CANDIDATE_K):
    # Обучаем на WikIR train
    train_scored = score_candidate_pairs_biencoder(
        model_name=BIENC_MODEL,
        queries_df=wikir_train_queries,
        docs_df=wikir_docs,
        candidates_df=top_k_per_query(wikir_bm25_train, candidate_k),
        batch_size=BATCH_SIZE,
        device=DEVICE,
        output_col="dense_score",
        cache_dir=EMBED_DIR,
    )
    train_scored = extract_lexical_features(train_scored, wikir_docs, wikir_train_queries)
    model = train_catboost_ltr(train_scored, wikir_train_qrels)

    rows = []
    for dataset_name, docs_df, queries_df, qrels_df, bm25_run in [
        ("WikIR test", wikir_docs, wikir_test_queries, wikir_test_qrels, wikir_bm25_test),
        ("MIRAGE test", mirage_test_docs, mirage_test_queries, mirage_test_qrels, mirage_bm25_test),
    ]:
        scored = score_candidate_pairs_biencoder(
            model_name=BIENC_MODEL,
            queries_df=queries_df,
            docs_df=docs_df,
            candidates_df=top_k_per_query(bm25_run, candidate_k),
            batch_size=BATCH_SIZE,
            device=DEVICE,
            output_col="dense_score",
            cache_dir=EMBED_DIR,
        )
        scored = extract_lexical_features(scored, docs_df, queries_df)
        tag = dataset_name.split()[0].lower()
        run = apply_catboost_ltr(model, scored, run_name=f"catboost_ltr_{tag}", top_k=20)
        run.to_csv(RUNS_DIR / f"catboost_ltr__{tag}.csv", index=False)
        rows.append({"dataset": dataset_name, "method": "CatBoost LTR", **evaluate_run(run, qrels_df)})

    return model, pd.DataFrame(rows)


catboost_model, catboost_ltr_df = run_catboost_ltr()
display(catboost_ltr_df)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11656.82it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7520.47it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8447.33it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
---------------------

,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,WikIR test,CatBoost LTR,0.49000,0.163000,0.11200,0.137992,0.306808
1,MIRAGE test,CatBoost LTR,0.64465,0.120079,0.06004,0.779734,0.837945


## Итоговая таблица

Собираем в одну таблицу результаты для `WikIR test` и `MIRAGE test`:
BM25 из HW2, dense retrieval, cross-encoder reranking, mixture и CatBoost LTR.

Основная метрика для сравнения — **nDCG@20**.


In [17]:
from IPython.display import display, Markdown

metric_cols = ["P@1", "P@10", "P@20", "MAP@20", "nDCG@20"]
metric_fmt = {col: "{:.4f}" for col in metric_cols}

bm25_tbl = (
    hw2_bm25_original_df[["dataset", "method"] + metric_cols]
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)

dense_tbl = dense_df.rename(columns={"model": "method"}).copy()
dense_tbl["method"] = "Dense " + dense_tbl["method"].astype(str)
dense_tbl = (
    dense_tbl[["dataset", "method"] + metric_cols]
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)

rerank_tbl = rerank_df.copy()
rerank_tbl["method"] = "CE k=" + rerank_tbl["k"].astype(str)
rerank_tbl = (
    rerank_tbl[["dataset", "method"] + metric_cols]
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)

mixture_tbl = (
    mixture_df[["dataset", "method"] + metric_cols]
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)

catboost_tbl = (
    catboost_ltr_df[["dataset", "method"] + metric_cols]
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)

summary_df = (
    pd.concat(
        [bm25_tbl, dense_tbl, rerank_tbl, mixture_tbl, catboost_tbl],
        ignore_index=True,
    )
    .sort_values(["dataset", "method"])
    .reset_index(drop=True)
)

tables = {
    "BM25": bm25_tbl,
    "Dense": dense_tbl,
    "Cross-Encoder": rerank_tbl,
    "Mixture": mixture_tbl,
    "CatBoost LTR": catboost_tbl,
    "Summary": summary_df,
}

for title, df in tables.items():
    display(Markdown(f"### {title}"))
    display(df.style.format(metric_fmt))


### BM25

,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,HW2 BM25 original,0.4967,0.1201,0.0600,0.6759,0.7598
1,WikIR test,HW2 BM25 original,0.2300,0.1320,0.0950,0.0975,0.2161


### Dense

,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,Dense all-MiniLM-L6-v2,0.6222,0.1181,0.0593,0.7577,0.8194
1,MIRAGE test,Dense multi-qa-MiniLM-L6-cos-v1,0.6380,0.1176,0.0594,0.7695,0.8277
2,WikIR test,Dense all-MiniLM-L6-v2,0.6400,0.1850,0.1265,0.1216,0.3574
3,WikIR test,Dense multi-qa-MiniLM-L6-cos-v1,0.6300,0.1840,0.1215,0.1223,0.3547


### Cross-Encoder

...

,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,BM25,0.4967,0.1201,0.0600,0.6759,0.7598
1,MIRAGE test,Dense,0.6631,0.1201,0.0600,0.7953,0.8492
2,MIRAGE test,Mixture α=0.10,0.6909,0.1201,0.0600,0.8103,0.8605
3,WikIR test,BM25,0.2300,0.1330,0.0945,0.0976,0.2160
4,WikIR test,Dense,0.4700,0.1480,0.1045,0.1286,0.2990
5,WikIR test,Mixture α=0.10,0.5100,0.1580,0.1065,0.1372,0.3082


### CatBoost LTR

,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,CatBoost LTR,0.6446,0.1201,0.0600,0.7797,0.8379
1,WikIR test,CatBoost LTR,0.4900,0.1630,0.1120,0.1380,0.3068


### Summary

,dataset,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,MIRAGE test,BM25,0.4967,0.1201,0.0600,0.6759,0.7598
1,MIRAGE test,CE k=5,0.8428,0.1201,0.0600,0.9025,0.9291
2,MIRAGE test,CatBoost LTR,0.6446,0.1201,0.0600,0.7797,0.8379
3,MIRAGE test,Dense,0.6631,0.1201,0.0600,0.7953,0.8492
4,MIRAGE test,Dense all-MiniLM-L6-v2,0.6222,0.1181,0.0593,0.7577,0.8194
5,MIRAGE test,Dense multi-qa-MiniLM-L6-cos-v1,0.6380,0.1176,0.0594,0.7695,0.8277
6,MIRAGE test,HW2 BM25 original,0.4967,0.1201,0.0600,0.6759,0.7598
7,MIRAGE test,Mixture α=0.10,0.6909,0.1201,0.0600,0.8103,0.8605
8,WikIR test,BM25,0.2300,0.1330,0.0945,0.0976,0.2160
9,WikIR test,CE k=10,0.4900,0.1270,0.0635,0.1062,0.2364


In [18]:
for dataset_name, part in summary_df.groupby("dataset", sort=False):
    display(Markdown(f"## {dataset_name}"))
    display(part[["method"] + metric_cols].reset_index(drop=True).style.format(metric_fmt))


## MIRAGE test

,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,BM25,0.4967,0.1201,0.0600,0.6759,0.7598
1,CE k=5,0.8428,0.1201,0.0600,0.9025,0.9291
2,CatBoost LTR,0.6446,0.1201,0.0600,0.7797,0.8379
3,Dense,0.6631,0.1201,0.0600,0.7953,0.8492
4,Dense all-MiniLM-L6-v2,0.6222,0.1181,0.0593,0.7577,0.8194
5,Dense multi-qa-MiniLM-L6-cos-v1,0.6380,0.1176,0.0594,0.7695,0.8277
6,HW2 BM25 original,0.4967,0.1201,0.0600,0.6759,0.7598
7,Mixture α=0.10,0.6909,0.1201,0.0600,0.8103,0.8605


## WikIR test

,method,P@1,P@10,P@20,MAP@20,nDCG@20
0,BM25,0.2300,0.1330,0.0945,0.0976,0.2160
1,CE k=10,0.4900,0.1270,0.0635,0.1062,0.2364
2,CE k=100,0.5900,0.1680,0.1165,0.1475,0.3309
3,CE k=20,0.5300,0.1460,0.0945,0.1295,0.2818
4,CE k=50,0.5700,0.1550,0.1095,0.1408,0.3156
5,CatBoost LTR,0.4900,0.1630,0.1120,0.1380,0.3068
6,Dense,0.4700,0.1480,0.1045,0.1286,0.2990
7,Dense all-MiniLM-L6-v2,0.6400,0.1850,0.1265,0.1216,0.3574
8,Dense multi-qa-MiniLM-L6-cos-v1,0.6300,0.1840,0.1215,0.1223,0.3547
9,HW2 BM25 original,0.2300,0.1320,0.0950,0.0975,0.2161
